In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from torch import nn
import numpy as np


In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)
print(torch.cuda.get_arch_list())


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
window_size = 30
pad_value = 0

In [ ]:
exclude_rgb_vis=False
exclude_weather_features=True
exclude_canopy_temp=True
dataset_name = "stratified_train_test_datasets_v4_all_interpolated.xlsx"

In [ ]:
import os
import random

seed = 23
def reset_seed_all(_seed):
    os.environ["PYTHONHASHSEED"] = str(_seed)
    random.seed(_seed)

    torch.manual_seed(_seed)
    np.random.seed(_seed)

    torch.cuda.manual_seed_all(_seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

    g = torch.Generator()
    g.manual_seed(_seed)
    return g

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [ ]:
from features import *

vi_features = get_vi_features(exclude_rgb_vis=exclude_rgb_vis)
weather_features = get_weather_features(exclude=exclude_weather_features)
canopy_temp_features = get_canopy_temp_features(exclude=exclude_canopy_temp)
features = vi_features + weather_features + canopy_temp_features

output_variable = get_output_variable()


In [ ]:
# Loading saved training test
train_df = pd.read_excel(f"data/{dataset_name}", sheet_name='Train')
test_df = pd.read_excel(f"data/{dataset_name}", sheet_name='Test')

sns.kdeplot(train_df[output_variable], label='Train')
sns.kdeplot(test_df[output_variable], label='Test')
plt.legend()
plt.title("Train vs Test Yield Distribution")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

scaler_X = StandardScaler()
scaler_y = RobustScaler()

# Fit on training features and yield only
scaler_X.fit(train_df[features])
scaler_y.fit(train_df[[output_variable]])

train_df_scaled = train_df.copy()
test_df_scaled  = test_df.copy()

# --- Scaled features and target variable---
train_df_scaled[features] = scaler_X.transform(train_df[features])
train_df_scaled[output_variable] = scaler_y.transform(train_df[[output_variable]])

test_df_scaled[features] = scaler_X.transform(test_df[features])
test_df_scaled[output_variable] = scaler_y.transform(test_df[[output_variable]])

In [ ]:

num_na_rows = train_df_scaled.isna().any(axis=1).sum()
print(f"train_df_scaled Rows with at least one NaN: {num_na_rows}")

num_na_rows = test_df_scaled.isna().any(axis=1).sum()
print(f"test_df_scaled Rows with at least one NaN: {num_na_rows}")


In [ ]:

def train_data(model, model_train_loader, num_epochs, learning_rate, weight_decay, hyper_param_criterion_method, file_save_name, stop_early = False):
    best_train_loss  = float('inf')
    patience = 20
    counter = 0

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )
    if hyper_param_criterion_method == "MSE":
        criterion = nn.MSELoss()
    else:
        criterion = nn.SmoothL1Loss(beta=1)
        # criterion = nn.SmoothL1Loss(beta=1.0, reduction="none")

    T = 46  # e.g., 45
    total_loss = []
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        for X_batch, y_batch, lengths in model_train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            lengths = lengths.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch, lengths)

            # loss_per_sample = criterion(y_pred, y_batch)
            loss = criterion(y_pred, y_batch)

            # weights = lengths.float() / T                 # shape: (batch,)
            # weights = weights / weights.mean()
            # loss = (weights * loss_per_sample).mean()

            loss.backward()
            # Gradient clipping (prevents exploding gradients)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)

        avg_loss = train_loss / len(model_train_loader.dataset)
        total_loss.append(avg_loss)

        # print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {avg_loss:.5f}")

        if train_loss < best_train_loss:
            best_train_loss = train_loss
            if file_save_name != "":
                torch.save(model.state_dict(), file_save_name)
            counter = 0
        elif stop_early:
            counter += 1
            if counter >= patience:
                print("Early stopping triggered.")
                break
    return total_loss


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


def evaluate_model(model, val_loader, output_variable_scaler, do_inverse_transform = False, plot_pred_vs_true = False):

    model.eval()  # set model to evaluation mode
    y_true, y_pred = [], []

    with torch.no_grad():
        for X_batch, y_batch, lengths in val_loader:
            X_batch = X_batch.to(device)
            lengths = lengths.to(device)
            y_hat = model(X_batch, lengths)

            y_true.append(y_batch.cpu())
            y_pred.append(y_hat.cpu())

    y_true = torch.cat(y_true).numpy()
    y_pred = torch.cat(y_pred).numpy()

    if do_inverse_transform:
        y_true = output_variable_scaler.inverse_transform(y_true)
        y_pred = output_variable_scaler.inverse_transform(y_pred)

    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)

    if plot_pred_vs_true:
        plt.figure(figsize=(8,6))
        plt.scatter(y_true, y_pred, alpha=0.7)
        plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'k--')
        plt.xlabel("Actual Yield")
        plt.ylabel("Predicted Yield")
        plt.title("GRU Predictions vs Actual")
        plt.show()

    return r2, mse, mae

In [ ]:
from helper import filter_window_data_by_length

total_n_days = 46 # (from may 1st to June 15th)

def get_results_by_day():
    all_test_r2 = []
    all_test_mse = []
    all_test_mae = []
    for day in range(1, total_n_days + 1):
        # print(f"day: {day}")

        x_train_scaled_filtered, y_train_scaled_filtered, lengths_train_filtered = filter_window_data_by_length(
            X_train_scaled, y_train_scaled, valid_length_train, no_days_train, day
        )

        # test
        x_test_scaled_filtered, y_test_scaled_filtered, lengths_test_filtered = filter_window_data_by_length(
            X_test_scaled, y_test_scaled, valid_length_test, no_days_test, day
        )

        train_dataset_filtered = VIDataset(x_train_scaled_filtered, y_train_scaled_filtered, lengths_train_filtered)
        test_dataset_filtered = VIDataset(x_test_scaled_filtered, y_test_scaled_filtered, lengths_test_filtered)

        train_loader_filtered = DataLoader(train_dataset_filtered, batch_size=best_gru_params['batch_size'],
                                           shuffle=True)
        test_loader_filtered = DataLoader(test_dataset_filtered, batch_size=best_gru_params['batch_size'],
                                          shuffle=False)

        # Training metrics
        r2_train_filtered, mse_train_filtered, mae_train_filtered = evaluate_model(best_model, train_loader_filtered,
                                                                                   output_variable_scaler=scaler_y,
                                                                                   do_inverse_transform=True,
                                                                                   plot_pred_vs_true=False)
        # Test metrics
        r2_test_filtered, mse_test_filtered, mae_test_filtered = evaluate_model(best_model, test_loader_filtered,
                                                                                output_variable_scaler=scaler_y,
                                                                                do_inverse_transform=True,
                                                                                plot_pred_vs_true=False)

        # print(
        #     f'Train R²: {r2_train_filtered:.4f}, Train MSE: {mse_train_filtered:.4f}, Train MAE: {mae_train_filtered:.4f}')
        # print(f'Test R²: {r2_test_filtered:.4f}, Test MSE: {mse_test_filtered:.4f}, Test MAE: {mae_test_filtered:.4f}')
        all_test_r2.append(r2_test_filtered)
        all_test_mse.append(mse_test_filtered)
        all_test_mae.append(mae_test_filtered)

    return all_test_r2, all_test_mse, all_test_mae


K-fold cross validation

In [ ]:
# from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# kf = KFold(n_splits=5, shuffle=True, random_state=42)
plot_ids = train_df['plot_id'].unique()
yield_per_plot = train_df.groupby('plot_id')[output_variable].first().values

y_bins = pd.qcut(yield_per_plot, q=5, labels=False)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)


In [ ]:
from helper import make_progressive_windows
from vi_dataset import VIDataset
from model_definitions import  GRUModel
from torch.utils.data import DataLoader

def k_fold_cross_validation(trial_id, rand_seed, batch_size, k_fold_epochs, hidden_size, num_layers, dropout, bidirectional, learning_rate, weight_decay, hyper_param_criterion_method, early_stopping = True):

    r2_scores = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(plot_ids, y_bins)):
        torch_gen = reset_seed_all(rand_seed)
        print(f"Fold {fold+1}")

        train_plots = plot_ids[train_idx]
        val_plots   = plot_ids[val_idx]

        train_df_fold = train_df[train_df['plot_id'].isin(train_plots)]
        val_df_fold   = train_df[train_df['plot_id'].isin(val_plots)]

        # Scaling
        scaler_X_fold = StandardScaler()
        scaler_y_fold = RobustScaler()

        scaler_X_fold.fit(train_df_fold[features])
        scaler_y_fold.fit(train_df_fold[[output_variable]])

        train_df_fold_scaled = train_df_fold.copy()
        val_df_fold_scaled  = val_df_fold.copy()

        train_df_fold_scaled[features] = scaler_X_fold.transform(train_df_fold[features])
        train_df_fold_scaled[output_variable] = scaler_y_fold.transform(train_df_fold[[output_variable]])

        val_df_fold_scaled[features] = scaler_X_fold.transform(val_df_fold[features])
        val_df_fold_scaled[output_variable] = scaler_y_fold.transform(val_df_fold[[output_variable]])

        X_train_fold, y_train_fold, valid_length_train_fold, no_days_train_fold  = make_progressive_windows(
            dataframe=train_df_fold_scaled,
            features=features,
            output_variable=output_variable,
            window_size=window_size,
            pad_value=pad_value)

        X_val_fold, y_val_fold, valid_length_test_fold, no_days_test_fold= make_progressive_windows(
            dataframe=val_df_fold_scaled,
            features=features,
            output_variable=output_variable,
            window_size=window_size,
            pad_value=pad_value)

        train_dataset_fold = VIDataset(X_train_fold, y_train_fold, valid_length_train_fold)
        val_dataset_fold = VIDataset(X_val_fold, y_val_fold, valid_length_test_fold)

        train_loader_fold  = DataLoader(train_dataset_fold, batch_size=batch_size, shuffle=True, num_workers =0, worker_init_fn=seed_worker, generator=torch_gen)
        val_loader_fold    = DataLoader(val_dataset_fold, batch_size=batch_size, shuffle=False,
                                        num_workers =0, worker_init_fn=seed_worker, generator=torch_gen)

        model = GRUModel(input_size = len(features), hidden_size = hidden_size, num_layers = num_layers,
                         output_size= 1, dropout=dropout, bidirectional = bidirectional).to(device)
        train_data(
                model = model,
                model_train_loader= train_loader_fold,
                num_epochs = k_fold_epochs,
                learning_rate = learning_rate,
                weight_decay = weight_decay,
                hyper_param_criterion_method = hyper_param_criterion_method,
                file_save_name=f"best_model_hyper_param{trial_id}_{fold}.pth",
                stop_early=early_stopping
            )

        r2, mse, mae = evaluate_model(
            model = model,
            val_loader = val_loader_fold,
            output_variable_scaler=scaler_y_fold)
        print(f'Test R²: {r2:.4f}, Test MSE: {mse:.4f}, Test MAE: {mae:.4f}')

        r2_scores.append(r2)
        print(f" Trial: {trial_id}, Fold {fold+1} R²: {r2:.4f}")

    mean_r2 = np.mean(r2_scores)
    print(f"\n Trial: {trial_id}, Mean R² across folds: {mean_r2:.4f}")
    return mean_r2

In [ ]:

# search_space = {
#     "hidden_size": [8, 16, 32 , 64],
#     "num_layers": [1, 2, 3],
#     "dropout": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
#     "lr":  [0.00001, 0.0001, 0.001, 0.01],
#     "batch_size": [8, 16, 32, 64],
# }

tuning_epochs = 1000

def objective(trial):
    hidden_size = trial.suggest_categorical('hidden_size', [8, 16])
    num_layers = trial.suggest_categorical('num_layers', [1, 2, 3])
    dropout = trial.suggest_float('dropout', 0.0,  0.5)
    lr = trial.suggest_float('lr', 1e-4,  1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-10, 1e-3, log=True)
    # bidirectional = trial.suggest_categorical('bidirectional', [True, False])
    batch_size = trial.suggest_categorical('batch_size', [8, 16, 32, 64])

    mean_r2 = k_fold_cross_validation(
        trial_id=trial.number,
        rand_seed=seed,
        batch_size= batch_size,
        k_fold_epochs=tuning_epochs,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
        bidirectional=False,
        learning_rate=lr,
        weight_decay=weight_decay,
        hyper_param_criterion_method="L1Loss")
    return mean_r2


In [ ]:
import optuna

sampler = optuna.samplers.TPESampler(seed=seed)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=100)

print(f"Best R²: {study.best_value:.4f}")
print("Best hyperparameters:", study.best_params)


In [ ]:
# window size = 30
# Dataset: v3_interpolated , seed = 46, and reproducible
# Features: Only VIs
# Masking/Length in GRU: False;  Make sure GRU model does not use lengths in the forward function
# Pad = 0
# criterion_method = L1 Loss
best_gru_params = {'hidden_size': 8, 'num_layers': 1, 'dropout': 0.40004704498537735, 'lr': 0.00011086801175787916, 'weight_decay': 2.0841640770116792e-07, 'batch_size': 32}

In [ ]:
from helper import make_progressive_windows
from vi_dataset import VIDataset
from sklearn.metrics import r2_score
from torch.utils.data import DataLoader

X_train_scaled, y_train_scaled, valid_length_train, no_days_train  = make_progressive_windows(
            dataframe=train_df_scaled,
            features=features,
            output_variable=output_variable,
            window_size=window_size,
            pad_value=pad_value)

X_test_scaled, y_test_scaled, valid_length_test, no_days_test = make_progressive_windows(
    dataframe=test_df_scaled,
    features=features,
    output_variable=output_variable,
    window_size=window_size,
    pad_value=pad_value)


In [ ]:
import csv
from plots import plot_r2, plot_mse

epochs = 1000
criterion_method="L1Loss"

all_mean_r2 = []
all_mae = []
all_mse = []
all_max_r2 = []

seeds_1 = [46, 36, 88, 56, 66]
seeds_2 = [23, 53, 68, 83, 93]
for rand_seed in range(5):
    # rand_seed = (seed + (i*15))
    # if i == 0:
    #     rand_seed = seed
    # else:
    rand_seed = random.randint(1, 200)


    print("--------------  Random Seed: ", rand_seed)

    g = reset_seed_all(rand_seed)

    train_dataset = VIDataset(X_train_scaled, y_train_scaled, valid_length_train)
    test_dataset = VIDataset(X_test_scaled, y_test_scaled, valid_length_test)

    train_loader  = DataLoader(train_dataset, batch_size=best_gru_params['batch_size'], shuffle=True, num_workers =0, worker_init_fn=seed_worker, generator=g)
    test_loader    = DataLoader(test_dataset, batch_size=best_gru_params['batch_size'], shuffle=False, num_workers =0, worker_init_fn=seed_worker, generator=g)

    best_model = GRUModel(
        input_size=len(features),
        hidden_size=best_gru_params['hidden_size'],
        num_layers=best_gru_params['num_layers'],
        output_size=1,
        dropout=best_gru_params['dropout'],
        bidirectional=False).to(device)

    train_data(
        model = best_model,
        model_train_loader= train_loader,
        num_epochs = epochs,
        learning_rate = best_gru_params['lr'],
        weight_decay = best_gru_params['weight_decay'],
        hyper_param_criterion_method = criterion_method,
        file_save_name=f"best_model_hyper_param999_ {rand_seed}.pth",
        stop_early=True
    )

    # best_model.load_state_dict(torch.load("best_gru_models/best_model_hyper_param_14.pth", map_location=device))

    # Training metrics
    r2_train, mse_train, mae_train = evaluate_model(best_model, train_loader, output_variable_scaler=scaler_y, do_inverse_transform=True, plot_pred_vs_true=False)
    # Test metrics
    r2_test, mse_test, mae_test = evaluate_model(best_model, test_loader, output_variable_scaler=scaler_y, do_inverse_transform=True, plot_pred_vs_true=False)

    test_r2, test_mse, test_mae = get_results_by_day()

    print(f'Train R²: {r2_train:.4f}, Train MSE: {mse_train:.4f}, Train MAE: {mae_train:.4f}')
    print(f'Test R²: {r2_test:.4f}, Test MSE: {mse_test:.4f}, Test MAE: {mae_test:.4f}, Max test R²: {np.max(test_r2):.4f}')

    # plot_r2(test_r2, total_n_days)
    # plot_mse(test_mse, total_n_days)

    # START_DAY = 122 # May 1st
    # gre_days = np.arange(START_DAY, START_DAY + total_n_days)


    # with open(f'results/best_gru_model_14_seed_{rand_seed}_performance.csv', 'w', newline='') as f:
    #     writer = csv.writer(f)
    #     writer.writerow(['Gregorian day', 'R2', 'MSE', 'MAE'])  # header
    #     for a, b, c, d in zip(gre_days, test_r2, test_mse, test_mae):
    #         writer.writerow([a, b,c,d])


    all_mean_r2.append(r2_test)
    all_mae.append(mae_test)
    all_mse.append(mse_test)
    all_max_r2.append(np.max(test_r2))



In [ ]:
    print("Mean of mean r2: ", np.mean(all_mean_r2))
    print("Std of mean r2: ", np.std(all_mean_r2))

    print("Mean of mean mse: ", np.mean(all_mse))
    print("Std of mean mse: ", np.std(all_mse))

    print("Mean of mean mae: ", np.mean(all_mae))
    print("Std of mean mae: ", np.std(all_mae))

    print("Mean of max r2: ", np.mean(all_max_r2))
    print("Std of max r2: ", np.std(all_max_r2))

In [ ]:
from train_eval_functions import evaluate_model

state_dict = torch.load("best_model_gru_kfold_1.pth", map_location=device)
best_model.load_state_dict(state_dict)

# Training metrics
r2_train, mse_train, mae_train = evaluate_model(best_model, train_loader, output_variable_scaler=scaler_y, do_inverse_transform= True, plot_pred_vs_true=True)
# Test metrics
r2_test, mse_test, mae_test = evaluate_model(best_model, test_loader, output_variable_scaler=scaler_y, do_inverse_transform= True, plot_pred_vs_true=True)

print(f'Train R²: {r2_train:.4f}, Train MSE: {mse_train:.4f}')
print(f'Test R²: {r2_test:.4f}, Test MSE: {mse_test:.4f}')


In [ ]:
# save model
torch.save(best_model, "best_gru_models/best_model_full_1.pth")

# save Scalers for future predictions
import joblib

joblib.dump(scaler_X, "best_gru_models/scalers/scaler_X_1.save")
joblib.dump(scaler_y, "best_gru_models/scalers/scaler_y_1.save")